# B0 training on Kaggle: fixed SfM subset-100, VGG16 + GeM

This notebook uses **only the already attached inputs**, never downloads datasets, never opens the full SfM backup for training, and never resamples supervision. Expected subset: seed 42; train **100 clusters / 15,953 images / 34,339 pairs**; validation **20 clusters / 739 images / 181 pairs**.

Model: shared VGG16 features → scalar GeM → L2 normalization → 512-D descriptor. Contrastive tuples contain q + p + five negatives from distinct clusters. Evaluation is **legacy Oxford5k/Paris6k, single scale [1], no whitening and no query expansion**.

Run cells in order. Smoke and full training are disabled independently by default. The provisional smoke configuration is resolved. Inspect preflight results, then enable smoke only; full-run choices remain separate. Enable full training in a separate pass. An OOM or mining failure stops the run; this notebook never silently reduces settings.

Source reviewed: origin/main `2f1aa489ec8c247cef2c0521e2c8575fef00a8f9`; origin/B0 `747166e627e514842f5c5be531f4e0f73612cc90`; data preparation `6e50faaaadb8101838e6da2e98c0681d6338a474`. The embedded source is an exact B0 snapshot; its only cirtorch changes relative to main are Pillow LANCZOS and NumPy isin compatibility substitutions. See the final deviation ledger.

The authoring environment has no /kaggle/input mounts. Actual attached paths are inspected below at Kaggle runtime; no dataset slug is assumed. No training or evaluation results are bundled.


## 1. Environment and GPU checks
Use Kaggle's preinstalled compatible torch/torchvision pair. **Do not install the historical requirements.txt or run pip install .** A CUDA GPU is required by the original implementation. One GPU is used even if two T4s are offered. Outputs are confined to a notebook-owned directory under /kaggle/working.


In [ ]:
import os, sys, json, pickle, random, math, time, hashlib, base64, zlib, re, gc
from pathlib import Path
from contextlib import contextmanager, redirect_stdout, redirect_stderr
from types import SimpleNamespace
import inspect, traceback
if sys.version_info < (3, 9):
    raise RuntimeError('Python >=3.9 is required for Path.is_relative_to and str.removeprefix.')
try:
    import numpy as np
    import PIL
    from PIL import Image
    import torch
    import torchvision
    from torchvision import transforms
except (ImportError, RuntimeError, OSError) as error:
    raise RuntimeError('Kaggle dependency import failed. Use a compatible preinstalled torch/torchvision pair; '
                       'do not install the historical repository requirements. Details: ' + str(error)) from error
required_apis = {
    'Pillow Image.Resampling.LANCZOS': hasattr(Image, 'Resampling') and hasattr(Image.Resampling, 'LANCZOS'),
    'NumPy isin': callable(getattr(np, 'isin', None)),
    'torch.load(weights_only=...)': 'weights_only' in inspect.signature(torch.load).parameters,
    'Optimizer.register_step_pre_hook': callable(getattr(torch.optim.Optimizer, 'register_step_pre_hook', None)),
    'torchvision VGG16 constructor': callable(getattr(torchvision.models, 'vgg16', None)),
    'torchvision image backend API': callable(getattr(torchvision, 'get_image_backend', None)),
}
missing = [name for name, present in required_apis.items() if not present]
if missing:
    raise RuntimeError('Missing required compatibility APIs: ' + ', '.join(missing))
torchvision.set_image_backend('PIL')
print('Dependency/API preflight passed:', required_apis)

INPUT_ROOT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
if not INPUT_ROOT.is_dir() or not WORKING.is_dir():
    raise RuntimeError('Run on Kaggle with the three datasets attached and a GPU enabled.')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
torch.cuda.set_device(0)
ENVIRONMENT = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,
                   numpy=np.__version__, pillow=PIL.__version__, cuda=torch.version.cuda,
                   gpu=torch.cuda.get_device_name(0), gpu_bytes=torch.cuda.get_device_properties(0).total_memory)
print(json.dumps(ENVIRONMENT, indent=2))
print('Devices:', torch.cuda.device_count(), 'Using device 0 only')
print('Free working disk GB:', __import__('shutil').disk_usage(WORKING).free / 1e9)
WORKSPACE = WORKING / 'b0_sfm100_notebook'
WORKSPACE.mkdir(exist_ok=True)

# Shared logger is defined before smoke; all run logs stay inside the notebook workspace.
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, value):
        for stream in self.streams: stream.write(value)
        return len(value)
    def flush(self):
        for stream in self.streams: stream.flush()

@contextmanager
def logged(path):
    with Path(path).open('a', buffering=1) as log:
        with redirect_stdout(Tee(sys.stdout, log)), redirect_stderr(Tee(sys.stderr, log)):
            yield


## 2. Offline repository setup (pinned source snapshot)
The notebook carries the original B0 Python source and license as compressed JSON. It materializes that exact snapshot under its own workspace, validates every SHA-256, and imports it without installation or Git operations. Existing differing files cause failure; unrelated files are never deleted. The source commit and per-file hashes are saved with run records. A fresh kernel is required if cirtorch was imported from a different checkout.


In [ ]:
SOURCE_COMMIT = '747166e627e514842f5c5be531f4e0f73612cc90'
SOURCE_HASHES = {'cirtorch/__init__.py': 'ef6a810989ab31e2d49446d7faaa9c735f85d2fccfdedafb531f41a85b5841b5', 'cirtorch/datasets/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/datasets/datahelpers.py': '6e18ba2c48ba66f000d83e94187f5a258862d4f490c73de1f94481da0ef811b3', 'cirtorch/datasets/genericdataset.py': '76bb2bab4493f5cfcda0691006442178727c7625abcc2e4e8ab40d6561d332d3', 'cirtorch/datasets/testdataset.py': '13c305c090657ca5624615c72ae2e8eaea26995632da9169d0b379728de117f3', 'cirtorch/datasets/traindataset.py': '2e79fa6a05ac4ee153fab383af438f6d092585205ba6ed8b1a86ab20266da702', 'cirtorch/examples/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/examples/example_descriptor_extraction.py': '6f408acfa97edee2da6d338ff59d8bab2828a079c7b9edd7a969b51dcc5aac8f', 'cirtorch/examples/test.py': '56bc6037a7e9e91873ae3f64aefeab46f983bee1966266e241346617b52b3f7c', 'cirtorch/examples/test_e2e.py': 'e7ec88802bfba68878df91fbfb617ba0b9d3e224f3689b24b554f24d4e380016', 'cirtorch/examples/train.py': '7f0cbfd34c8075139e6225ec7db72312b97c5a0f2d7762ace9c473c928f33848', 'cirtorch/layers/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/layers/functional.py': 'c016ffb064d458c920c61add3f6d03f1f11ae2e89d1e06facf5854636e41019c', 'cirtorch/layers/loss.py': 'fbaa3d82e297cb77481f93fe49de10af30158a0f94f487dc80bca5888f82715d', 'cirtorch/layers/normalization.py': '9ae607b73af5b0112c83a38db26586447e95f92359a3ca7eb7c5fefa3340f076', 'cirtorch/layers/pooling.py': 'c22d6d6b72e45d7f2454e6ea1b263b7ad45f548b8e50b1914fcce92bd74f8c64', 'cirtorch/networks/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/networks/imageretrievalnet.py': 'ffc64fca4c1077987f40640eec90500f7f26b80f3a37703432572f21e8288a60', 'cirtorch/utils/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'cirtorch/utils/download.py': 'd5bf645ec5417457b4062dbf010cc8388d3310d779cbe70beffd3e8000e2b195', 'cirtorch/utils/download_win.py': '643261a89fe1ceaaff19cb9c5244af178d4dc833542f87e51042a0f09e3aba19', 'cirtorch/utils/evaluate.py': 'ced1d110dd19f3f887e75f18ac9120aaff8900f5d6b52dc8c45c66c823141fd0', 'cirtorch/utils/general.py': '626e9f916199f7906926485c60b0a197e2ab112ab74cc66393ed5e54a295ea21', 'cirtorch/utils/whiten.py': '05162c86e6d8deea75d0d814d5469fa660e3006d2c98f6193a441dd08fb50dc1', 'LICENSE': 'c31c66c3945689e8876c9bbdb360228d13e4a15d84a77452dbe2db0ab67558e7'}
SOURCE_PAYLOAD = '...'
REPO_ROOT = WORKSPACE / ('source_' + SOURCE_COMMIT[:12])
source_files = json.loads(zlib.decompress(base64.b64decode(SOURCE_PAYLOAD)))
assert set(source_files) == set(SOURCE_HASHES)
for relative, contents in source_files.items():
    rel = Path(relative)
    if rel.is_absolute() or '..' in rel.parts or not (relative == 'LICENSE' or relative.startswith('cirtorch/')):
        raise RuntimeError('Unsafe embedded source path: ' + relative)
    encoded = contents.encode()
    assert hashlib.sha256(encoded).hexdigest() == SOURCE_HASHES[relative]
    dest = REPO_ROOT / rel
    if dest.exists():
        if dest.is_symlink() or dest.read_bytes() != encoded:
            raise RuntimeError('Existing source differs: ' + str(dest))
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(encoded)
if 'cirtorch' in sys.modules:
    imported = Path(sys.modules['cirtorch'].__file__).resolve()
    if not imported.is_relative_to(REPO_ROOT.resolve()):
        raise RuntimeError('Restart kernel: cirtorch is already imported from ' + str(imported))
sys.path.insert(0, str(REPO_ROOT))
import cirtorch
from cirtorch.datasets import traindataset, datahelpers
from cirtorch.datasets.testdataset import configdataset
from cirtorch.networks import imageretrievalnet
from cirtorch.layers.loss import ContrastiveLoss
from cirtorch.layers.pooling import GeM
from cirtorch.utils import general
from cirtorch.utils.evaluate import compute_ap, compute_map
from cirtorch.examples import train as original_train
assert Path(cirtorch.__file__).resolve().is_relative_to(REPO_ROOT.resolve())
print('Pinned source:', SOURCE_COMMIT, cirtorch.__file__)


## 3. Inspect attached inputs and discover paths
Discovery walks the real mounts and prints candidate metadata, weights, and provenance files. It tests image-directory candidates against CID paths and benchmark gallery/query IDs. Ambiguous candidates fail with their paths; fill the corresponding override using a printed actual path. Overrides must still point inside /kaggle/input and satisfy validation. No guessed dataset slugs or dataset copies are used.


In [ ]:
# Optional overrides ONLY after inspecting actual printed mount paths.
PATH_OVERRIDES = dict(sfm_metadata=None, sfm_images=None, oxford_ground_truth=None,
                      oxford_images=None, paris_ground_truth=None, paris_images=None, weights=None)
file_index = {}
for mount in sorted(INPUT_ROOT.iterdir()):
    print('Attached:', mount)
    if mount.is_dir():
        for parent, dirs, files in os.walk(mount):
            for filename in files:
                file_index.setdefault(filename, []).append(Path(parent) / filename)
for filename in ['retrieval-SfM-120k.pkl', 'retrieval-SfM-120k-full.pkl',
                 'subset_provenance_k100_seed42.txt', 'gnd_oxford5k.pkl', 'gnd_paris6k.pkl']:
    print(filename, ':', [str(p) for p in file_index.get(filename, [])])
print('Weight candidates:', [str(p) for name, ps in file_index.items() if name.endswith('.pth') for p in ps])

def input_path(p):
    p = Path(p).resolve()
    if not p.is_relative_to(INPUT_ROOT.resolve()) or not p.exists():
        raise RuntimeError('Expected existing mounted input: ' + str(p))
    return p

def choose(key, candidates):
    candidates = sorted(set(input_path(p) for p in candidates))
    override = PATH_OVERRIDES[key]
    if override is not None:
        p = input_path(override)
        if p not in candidates:
            raise RuntimeError(f'{key}: override does not satisfy discovery checks: {p}; candidates={candidates}')
        return p
    if len(candidates) != 1:
        raise RuntimeError(f'{key}: expected one match, found {len(candidates)}: {candidates}. Set PATH_OVERRIDES.')
    return candidates[0]

SFM_PKL = choose('sfm_metadata', file_index.get('retrieval-SfM-120k.pkl', []))
assert SFM_PKL.name == 'retrieval-SfM-120k.pkl'
with SFM_PKL.open('rb') as f:
    SFM_DB = pickle.load(f)  # Trusted team-supplied mounted metadata only.
PROVENANCE_PATH = SFM_PKL.parent / 'subset_provenance_k100_seed42.txt'
FILTER_PATH = SFM_PKL.parent / 'needed_files.txt'
for p in [PROVENANCE_PATH, FILTER_PATH]:
    input_path(p)
WEIGHTS = choose('weights', file_index.get('imagenet-caffe-vgg16-features-d369c8e.pth', []))
print('Active metadata:', SFM_PKL)
print('Full backup is ignored; whitening metadata is unused.')


## 4. Validate the fixed subset-100 metadata
Checks exact user-specified counts, all paired indices, unique CIDs, query/positive cluster consistency, and disjoint train/validation CIDs and cluster IDs. Checks provenance seed/counts and the complete extraction filter against the active metadata. **Nothing is resampled, rewritten, or restored from the full backup.** Counts plus a provenance label do not independently prove the historical draw; hashes and selected IDs identify the exact supplied subset for future comparison.


In [ ]:
EXPECTED_SUBSET = {'train': (15953, 100, 34339), 'val': (739, 20, 181)}
assert isinstance(SFM_DB, dict) and all(part in SFM_DB for part in EXPECTED_SUBSET)
SUBSET_SUMMARY = {}
for part, expected in EXPECTED_SUBSET.items():
    d = SFM_DB[part]
    assert all(k in d for k in ['cids', 'cluster', 'qidxs', 'pidxs']), part
    cids, clusters, qs, ps = (d[k] for k in ['cids', 'cluster', 'qidxs', 'pidxs'])
    assert len(cids) == len(clusters) and len(qs) == len(ps), part
    actual = (len(cids), len(set(clusters)), len(qs))
    assert actual == expected, f'{part}: expected {expected}, got {actual}; refuse a different subset'
    assert len(set(cids)) == len(cids)
    assert all(isinstance(cid, str) and len(cid) >= 6 and Path(cid).name == cid
               and cid not in {'.', '..'} for cid in cids)
    assert all(isinstance(i, (int, np.integer)) and 0 <= i < len(cids) for i in list(qs) + list(ps))
    assert all(q != p and clusters[q] == clusters[p] for q, p in zip(qs, ps)), part
    SUBSET_SUMMARY[part] = dict(images=actual[0], clusters=actual[1], pairs=actual[2],
                                cluster_ids=sorted(int(c) if isinstance(c, np.integer) else c for c in set(clusters)))
assert not set(SFM_DB['train']['cids']) & set(SFM_DB['val']['cids']), 'CID leakage'
assert not set(SFM_DB['train']['cluster']) & set(SFM_DB['val']['cluster']), 'Cluster leakage'
ALL_CIDS = list(SFM_DB['train']['cids']) + list(SFM_DB['val']['cids'])
assert len(ALL_CIDS) == len(set(ALL_CIDS)) == 16692
provenance_text = PROVENANCE_PATH.read_text()
print(provenance_text)
assert re.search(r'num_clusters\s*=\s*100\b', provenance_text)
assert re.search(r'seed\s*=\s*42\b', provenance_text)
for part, (ni, nc, npairs) in EXPECTED_SUBSET.items():
    assert re.search(rf'{part}:\s*{nc}\s+clusters,\s*{ni:,}\s+images,\s*{npairs:,}\s+pairs', provenance_text)
expected_filter = {f'{cid[-2:]}/{cid[-4:-2]}/{cid[-6:-4]}/{cid}' for cid in ALL_CIDS}
actual_filter = {line.removeprefix('./') for line in FILTER_PATH.read_text().splitlines() if line.strip()}
assert actual_filter == expected_filter, 'needed_files.txt differs from active metadata'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

DATASET_IDENTITY = dict(subset_seed=42, partitions=SUBSET_SUMMARY,
                       metadata_sha256=sha256_file(SFM_PKL), provenance_sha256=sha256_file(PROVENANCE_PATH),
                       filter_sha256=sha256_file(FILTER_PATH), weights_sha256=sha256_file(WEIGHTS))
print(json.dumps(SUBSET_SUMMARY, indent=2))


## 5. Image paths and benchmark integrity
Every metadata-referenced SfM image and every benchmark gallery/query image must exist. All ground-truth indices and bounding boxes are checked. Every query crop is checked against actual image dimensions. By default **all referenced images are fully decoded** on CPU, not merely sampled. This can take several minutes; progress is printed. Unreferenced extra SfM archive images are not used or copied. A runtime fingerprint records paths/sizes and metadata hashes; image content checksums are not computed.


In [ ]:
# Infer SfM ims root from actual CID occurrences rather than a dataset slug.
first_cid = ALL_CIDS[0]
ims_candidates = []
for path in file_index.get(first_cid, []):
    root = path.parents[3]
    if Path(datahelpers.cid2filename(first_cid, str(root))) == path:
        if all(Path(datahelpers.cid2filename(cid, str(root))).is_file() for cid in ALL_CIDS):
            ims_candidates.append(root)
SFM_IMAGES = choose('sfm_images', ims_candidates)
BENCHMARKS = {}
for name, key, expected_images in [('oxford5k', 'oxford', 5063), ('paris6k', 'paris', 6412)]:
    gp = choose(key + '_ground_truth', file_index.get('gnd_' + name + '.pkl', []))
    with gp.open('rb') as f:
        gt = pickle.load(f)
    assert all(k in gt for k in ['imlist', 'qimlist', 'gnd']), name
    assert len(gt['imlist']) == expected_images and len(set(gt['imlist'])) == expected_images
    assert len(gt['qimlist']) == len(gt['gnd']) == 55
    ids = set(gt['imlist']) | set(gt['qimlist'])
    assert all(isinstance(i, str) and Path(i).name == i and i not in {'.', '..'} for i in ids)
    sample_id = gt['imlist'][0] + '.jpg'
    image_dirs = [p.parent for p in file_index.get(sample_id, [])
                  if all((p.parent / (i + '.jpg')).is_file() for i in ids)]
    jpg = choose(key + '_images', image_dirs)
    for qid, entry in zip(gt['qimlist'], gt['gnd']):
        assert all(k in entry for k in ['bbx', 'ok', 'junk'])
        assert not any(k in entry for k in ['easy', 'hard']), 'Revisited protocol is not B0'
        box = np.asarray(entry['bbx'], dtype=float)
        assert box.shape == (4,) and np.isfinite(box).all()
        x0, y0, x1, y1 = box
        assert 0 <= x0 < x1 and 0 <= y0 < y1, (qid, box)
        with Image.open(jpg / (qid + '.jpg')) as image:
            assert x1 <= image.width and y1 <= image.height, (qid, box, image.size)
            crop = image.crop(tuple(box))
            assert min(crop.size) > 0
        for field in ['ok', 'junk']:
            assert all(isinstance(i, (int, np.integer)) and 0 <= i < expected_images for i in entry[field])
        assert len(entry['ok']) > 0 and not set(entry['ok']) & set(entry['junk'])
    BENCHMARKS[name] = dict(ground_truth=gp, images=jpg, gnd=gt)
    DATASET_IDENTITY[name] = dict(ground_truth_sha256=sha256_file(gp), gallery=expected_images, queries=55)

referenced = [Path(datahelpers.cid2filename(cid, str(SFM_IMAGES))) for cid in ALL_CIDS]
for b in BENCHMARKS.values():
    referenced.extend(b['images'] / (i + '.jpg') for i in set(b['gnd']['imlist']) | set(b['gnd']['qimlist']))
file_digest = hashlib.sha256()
for index, path in enumerate(sorted(set(referenced)), 1):
    assert path.is_file() and path.stat().st_size > 0, path
    with Image.open(path) as image:
        image.convert('RGB').load()  # Forces decoding; do not enable truncated-image acceptance.
    file_digest.update((str(path.relative_to(INPUT_ROOT)) + ':' + str(path.stat().st_size) + '\n').encode())
    if index % 2000 == 0:
        print('Fully decoded:', index, '/', len(set(referenced)))
DATASET_IDENTITY['referenced_paths_sizes_sha256'] = file_digest.hexdigest()
print('All referenced images decoded successfully:', len(set(referenced)))
PATHS = dict(sfm_metadata=str(SFM_PKL), sfm_images=str(SFM_IMAGES), weights=str(WEIGHTS),
             **{name: dict(ground_truth=str(b['ground_truth']), images=str(b['images'])) for name, b in BENCHMARKS.items()})
print(json.dumps(PATHS, indent=2))

# Writable scaffold only; symlink targets remain on the read-only mounts.
DATA_ROOT = WORKSPACE / 'data'
def link_input(destination, source):
    destination = Path(destination)
    source = input_path(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_symlink():
        assert destination.resolve() == source, f'Existing link differs: {destination}'
    elif destination.exists():
        raise RuntimeError('Refusing to replace existing path: ' + str(destination))
    else:
        destination.symlink_to(source, target_is_directory=source.is_dir())
sfm_stage = DATA_ROOT / 'train' / 'retrieval-SfM-120k'
link_input(sfm_stage / 'retrieval-SfM-120k.pkl', SFM_PKL)
link_input(sfm_stage / 'ims', SFM_IMAGES)
for name, b in BENCHMARKS.items():
    link_input(DATA_ROOT / 'test' / name / ('gnd_' + name + '.pkl'), b['ground_truth'])
    link_input(DATA_ROOT / 'test' / name / 'jpg', b['images'])
link_input(DATA_ROOT / 'networks' / WEIGHTS.name, WEIGHTS)
# Update imported function bindings in memory; original source files are untouched.
for module in [general, traindataset, imageretrievalnet, original_train]:
    module.get_data_root = lambda: str(DATA_ROOT)
# Never call original_train.main(): it automatically invokes dataset downloaders.

# Defensive offline guard: no preparation path may invoke original dataset/weight downloaders.
def reject_download(*args, **kwargs):
    raise RuntimeError('Dataset/weight downloading is forbidden: use only attached Kaggle inputs.')
from cirtorch.utils import download as source_download
source_download.download_train = source_download.download_test = reject_download
original_train.download_train = original_train.download_test = reject_download
original_train.main = reject_download
imageretrievalnet.model_zoo.load_url = reject_download
assert (sfm_stage / 'retrieval-SfM-120k.pkl').resolve() == SFM_PKL
assert (sfm_stage / 'ims').resolve() == SFM_IMAGES
assert not (sfm_stage / 'retrieval-SfM-120k-full.pkl').exists()
print('Offline guard active; active metadata/images are read-only input links.')


## 6–8. Provisional smoke configuration, separate full choices, weights and GeM
The first smoke configuration is explicitly authorized: VGG16, scalar GeM p=3, image_size=512, five tuples (seven images per tuple), Adam LR=1e-6, weight_decay=1e-6, contrastive margin=.7 and training seed42. GeM p uses the original 10x LR=1e-5 and zero decay. The subset remains the supplied k100/seed42 metadata.

Other retained source defaults: accumulation1, workers8, SGD-only momentum=.9, evaluation image size1024, test frequency1, pre-epoch ExponentialLR gamma=exp(-.01), validation enabled and best-by-validation-loss selection. Smoke performs just one optimizer update and one validation tuple; it does not evaluate benchmarks or exercise the scheduler. Its pool is independently controlled by SMOKE_MINING_POOL_CAP=256, capped only by the 15,953 available train images. Validation mining uses all 739 images.

TRAIN_CHOICES is fully resolved for smoke. FULL_TRAIN_CHOICES and FULL_DECISION_NOTE remain separate and unresolved intentionally; a full run requires explicit resource decisions. Source defaults are reference values, not a full-run agreement. Every execution flag remains False. For a retry, change controls after cell13 or edit and rerun that cell; rerunning it resets all flags.

Preprocessing remains original RGB ToTensor and source mean/std; no BGR conversion, AMP, clipping, feature freezing, augmentation or parallel model is added. Seed42 differs from upstream seed0 and is documented as a provisional smoke choice. A 128-pixel initialization probe checks local Caffe loading and descriptor shape; it does not prove 512-pixel training memory viability.


In [ ]:
SOURCE_DEFAULTS = dict(image_size=1024, query_size=2000, pool_size=20000, epochs=100,
                       batch_size=5, update_every=1, optimizer='adam', lr=1e-6, weight_decay=1e-6,
                       momentum=0.9, loss_margin=0.7, gem_p=3.0, seed=0, workers=8,
                       test_freq=1, eval_image_size=1024, scheduler='upstream_pre_epoch',
                       selection='validation_loss', validation=True)
# User-authorized provisional smoke values; remaining values are verified source defaults.
# query_size=5 / epochs=1 are diagnostic workload limits, not a full-run experiment.
TRAIN_CHOICES = dict(SOURCE_DEFAULTS, image_size=512, query_size=5, pool_size=256,
                     epochs=1, batch_size=5, lr=1e-6, weight_decay=1e-6,
                     loss_margin=0.7, gem_p=3.0, seed=42)
SMOKE_MINING_POOL_CAP = 256  # Change this explicitly if sampled cluster diversity is insufficient.
DECISION_NOTE = ('User-authorized provisional smoke: VGG16-GeM p3, 512px, five tuples, '
                 'LR/decay 1e-6, margin .7, seed42; one update and one validation tuple. '
                 'Adam, momentum .9 (SGD only), FP32, workers8, accumulation1, '
                 'pre-epoch exponential scheduler and legacy eval size1024 retain source defaults. '
                 'Smoke seed42 differs from upstream initialization/epoch seed0; no subset changes.')
# Full training cannot inherit a one-batch smoke budget. Resolve independently before opting in.
FULL_TRAIN_CHOICES = {key: None for key in SOURCE_DEFAULTS}
FULL_DECISION_NOTE = ''
RUN_ID = 'b0_sfm100_vgg16_gem'  # Use a new ID for a new experiment, not an accidental overwrite.
RESUME_CHECKPOINT = None  # Exact path to a trusted notebook-generated latest/best checkpoint.
RUN_SMOKE = False
RUN_FULL_TRAINING = False
RUN_FINAL_EVALUATION = False
EVALUATION_CHECKPOINT = None  # None selects this run's best.pt if it exists; never evaluates smoke weights.
print('Provisional smoke:', json.dumps(TRAIN_CHOICES, indent=2), 'pool cap=', SMOKE_MINING_POOL_CAP)
print('Unresolved full-run choices:', [key for key, value in FULL_TRAIN_CHOICES.items() if value is None])
print('SOURCE DEFAULTS (not a team agreement):', json.dumps(SOURCE_DEFAULTS, indent=2))

def resolved_config(purpose='smoke'):
    assert purpose in ['smoke', 'full']
    choices = TRAIN_CHOICES if purpose == 'smoke' else FULL_TRAIN_CHOICES
    note = DECISION_NOTE if purpose == 'smoke' else FULL_DECISION_NOTE
    missing = [key for key, value in choices.items() if value is None]
    if missing or not note.strip():
        raise RuntimeError(f'Resolve {purpose} choices={missing} and its decision note before execution.')
    cfg = dict(choices)
    if purpose == 'smoke':
        assert isinstance(SMOKE_MINING_POOL_CAP, int) and not isinstance(SMOKE_MINING_POOL_CAP, bool)
        assert SMOKE_MINING_POOL_CAP >= 6
        cfg.update(query_size=cfg['batch_size'], pool_size=min(SMOKE_MINING_POOL_CAP, 15953),
                   update_every=1, epochs=1)
    for key in ['image_size', 'query_size', 'pool_size', 'epochs', 'batch_size', 'update_every',
                'seed', 'workers', 'test_freq', 'eval_image_size']:
        assert isinstance(cfg[key], int) and not isinstance(cfg[key], bool), key
    assert cfg['image_size'] >= 32 and cfg['eval_image_size'] >= 32
    assert 1 <= cfg['batch_size'] <= cfg['query_size'] <= 34339
    assert 6 <= cfg['pool_size'] and cfg['epochs'] > 0 and cfg['update_every'] > 0
    assert cfg['seed'] >= 0 and cfg['workers'] >= 0 and cfg['test_freq'] > 0
    assert cfg['optimizer'] in ['adam', 'sgd']
    assert all(math.isfinite(cfg[k]) and cfg[k] > 0 for k in ['lr', 'loss_margin', 'gem_p'])
    assert math.isfinite(cfg['weight_decay']) and cfg['weight_decay'] >= 0
    assert math.isfinite(cfg['momentum']) and 0 <= cfg['momentum'] < 1
    assert cfg['scheduler'] == 'upstream_pre_epoch', 'Only the documented upstream scheduler policy is implemented'
    assert cfg['selection'] == 'validation_loss' and cfg['validation'] is True
    # Original accumulation discards a residual accumulation group. Fail rather than silently lose updates.
    assert (cfg['query_size'] // cfg['batch_size']) % cfg['update_every'] == 0
    assert cfg['batch_size'] <= 181, 'Validation must have at least one complete batch'
    return cfg

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    # Match source arithmetic; do not force a new deterministic-algorithm policy.

# Strict local feature-state loading: same init_network VGG16 feature topology,
# same load_state_dict as upstream, with local transport instead of load_url.
FEATURE_STATE = torch.load(WEIGHTS, map_location='cpu', weights_only=True)
assert isinstance(FEATURE_STATE, dict) and FEATURE_STATE
assert all(isinstance(k, str) and isinstance(v, torch.Tensor) for k, v in FEATURE_STATE.items())

def make_model(p=3.0):
    model = imageretrievalnet.init_network(dict(architecture='vgg16', pooling='gem', pretrained=False,
                    local_whitening=False, whitening=False, regional=False))
    model.features.load_state_dict(FEATURE_STATE, strict=True)
    with torch.no_grad():
        model.pool.p.fill_(p)
    assert isinstance(model.pool, GeM) and model.pool.p.numel() == 1
    assert model.meta['outputdim'] == 512 and model.whiten is None and model.lwhiten is None
    for k, tensor in model.features.state_dict().items():
        assert torch.equal(tensor.cpu(), FEATURE_STATE[k]), 'Pretrained load mismatch: ' + k
    return model.cuda()

# Initialization proof only: no optimizer updates, no mining, no full training.
seed_all(TRAIN_CHOICES['seed'])
probe_model = make_model(TRAIN_CHOICES['gem_p'])
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(
               mean=probe_model.meta['mean'], std=probe_model.meta['std'])])
probe = datahelpers.pil_loader(str(referenced[0]))
probe = datahelpers.imresize(probe, 128)  # Diagnostic-only forward size, never a training setting.
with torch.no_grad():
    descriptor = probe_model(transform(probe).unsqueeze(0).cuda())
assert descriptor.shape == (512, 1) and torch.isfinite(descriptor).all()
assert torch.allclose(descriptor.norm(dim=0), torch.ones(1, device='cuda'), atol=1e-5)
print(probe_model.meta_repr(), 'p=', probe_model.pool.p.item(), 'descriptor=', tuple(descriptor.shape))
del probe_model, descriptor, probe
gc.collect(); torch.cuda.empty_cache()


## 9. Original data loader, mining diagnostics, loss and optimizer
The original mining and training/validation loops remain embedded byte-for-byte and are called unchanged. Before mining, a separate CPU generator replays the two original permutations from the current RNG state solely to report the exact sampled pool's distinct clusters and the minimum eligible clusters across queries. This preview consumes no global randomness. Insufficient diversity fails clearly before descriptor extraction; there is no automatic resampling or negative substitution.

The smoke cap is configurable independently of full training. Original mining uses eight workers; workers controls the outer loaders only. Original drop_last=True is retained: full validation with batch5 processes180 of181 pairs, while smoke explicitly validates one tuple. Full validation mining uses all181 pairs and739 candidate images; smoke uses one pair and all739 images. The original train loader caps its candidate pool at15,953.


In [ ]:
def preview_mining_pool(dataset):
    # Original method draws query permutation, then image permutation, from CPU RNG.
    # Replay both on a separate generator: no global RNG consumption or pool resampling.
    global_state = torch.get_rng_state().clone()
    preview_rng = torch.Generator(device='cpu')
    preview_rng.set_state(global_state)
    selected_pairs = torch.randperm(len(dataset.qpool), generator=preview_rng)[:dataset.qsize].tolist()
    sampled_pool = torch.randperm(len(dataset.images), generator=preview_rng)[:dataset.poolsize].tolist()
    query_indices = [int(dataset.qpool[i]) for i in selected_pairs]
    pool_clusters = {dataset.clusters[i] for i in sampled_pool}
    eligible_counts = [len(pool_clusters - {dataset.clusters[q]}) for q in query_indices]
    assert torch.equal(global_state, torch.get_rng_state()), 'Diagnostics altered the original RNG'
    diagnostic = dict(partition=dataset.mode, sampled_images=len(sampled_pool),
                      distinct_pool_clusters=len(pool_clusters),
                      minimum_eligible_negative_clusters=min(eligible_counts), queries=len(query_indices))
    return diagnostic, query_indices

class CheckedTuplesDataset(traindataset.TuplesDataset):
    def create_epoch_tuples(self, net):
        diagnostic, expected_queries = preview_mining_pool(self)
        self.last_mining_diagnostic = diagnostic
        print('Sampled mining pool:', json.dumps(diagnostic))
        if diagnostic['minimum_eligible_negative_clusters'] < self.nnum:
            raise RuntimeError(f'Insufficient sampled pool diversity: {diagnostic}. '
                               'Increase SMOKE_MINING_POOL_CAP for smoke, or full pool_size for training; '
                               'the original pool is never resampled automatically.')
        try:
            distance = super().create_epoch_tuples(net)  # Exact embedded upstream algorithm.
        except IndexError as error:
            raise RuntimeError('Original mining exhausted the sampled pool. '
                               'Increase the corresponding pool setting explicitly; no resampling performed.') from error
        assert [int(q) for q in self.qidxs] == expected_queries, 'Original query draw differed from diagnostic replay'
        assert math.isfinite(distance)
        assert len(self.qidxs) == len(self.pidxs) == len(self.nidxs) == self.qsize
        for q, p, negatives in zip(self.qidxs, self.pidxs, self.nidxs):
            assert self.clusters[q] == self.clusters[p]
            assert len(negatives) == 5 and len({int(i) for i in negatives}) == 5
            assert len({self.clusters[i] for i in negatives}) == 5
            assert self.clusters[q] not in {self.clusters[i] for i in negatives}
        print(f'Mining checked: {self.mode}, tuples={self.qsize}, q+p+5=7, pool={self.poolsize}, mean_ndist={distance:.6f}')
        return distance

def make_loaders(cfg):
    train_ds = CheckedTuplesDataset('retrieval-SfM-120k', 'train', imsize=cfg['image_size'],
                    nnum=5, qsize=cfg['query_size'], poolsize=cfg['pool_size'], transform=transform)
    val_ds = CheckedTuplesDataset('retrieval-SfM-120k', 'val', imsize=cfg['image_size'],
                    nnum=5, qsize=float('inf'), poolsize=float('inf'), transform=transform)
    kwargs = dict(batch_size=cfg['batch_size'], num_workers=cfg['workers'], pin_memory=True,
                  drop_last=True, collate_fn=datahelpers.collate_tuples)
    train_loader = torch.utils.data.DataLoader(train_ds, shuffle=True, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, shuffle=False, **kwargs)
    print('Actual train queries/pool:', train_ds.qsize, train_ds.poolsize)
    print('Validation pairs/pool/processed:', val_ds.qsize, val_ds.poolsize, len(val_loader)*cfg['batch_size'])
    return train_loader, val_loader

def make_optimizer(model, cfg):
    groups = [{'params': model.features.parameters()},
              {'params': model.pool.parameters(), 'lr': cfg['lr'] * 10, 'weight_decay': 0}]
    if cfg['optimizer'] == 'adam':
        return torch.optim.Adam(groups, lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    return torch.optim.SGD(groups, lr=cfg['lr'], momentum=cfg['momentum'], weight_decay=cfg['weight_decay'])

def configure_original_loop(cfg):
    original_train.args = SimpleNamespace(update_every=cfg['update_every'], print_freq=10)

# Legacy arithmetic sanity checks; no models or benchmark forwards.
assert abs(compute_ap(np.array([2]), 1) - 1/6) < 1e-12
score, _, _, _ = compute_map(np.array([[0], [1], [2], [3]]), [{'ok': [3], 'junk': [0, 1]}])
assert abs(score - 0.25) < 1e-12


## 10. Separate smoke test — opt-in, logging and failure cleanup
Run cells3,5,7,9,11,13,15 first; then set RUN_SMOKE=True and execute cell17. Keep RUN_FULL_TRAINING and RUN_FINAL_EVALUATION False. Smoke uses512-pixel images, five seven-image tuples, one update and a configurable pool starting at256. Change SMOKE_MINING_POOL_CAP explicitly if diagnostics report fewer than five eligible negative clusters; the dataset itself is never changed.

Each attempt writes only a unique /kaggle/working/b0_sfm100_notebook/smoke/<timestamp>/ folder containing smoke.log, result.json and a smoke-only roundtrip.pt. Model and optimizer objects stay inside a function. try/except/finally removes hooks, clears upstream exception frames, releases GPU tensors and empties unused CUDA cache on failures as well as success. Failures raise a new clear error pointing to the log; no old activation traceback is retained. A CUDA context error may still require a kernel restart. Smoke weights are discarded and cannot overwrite or initialize full-run checkpoints.


In [ ]:
SMOKE_RESULT = None

def run_smoke_test(cfg):
    # Function scope and finally cleanup keep failed GPU objects out of notebook globals.
    smoke_dir = WORKSPACE / 'smoke' / str(time.time_ns())
    smoke_dir.mkdir(parents=True, exist_ok=False)
    assert not smoke_dir.is_relative_to(WORKSPACE / 'runs'), 'Smoke must not share full-run storage'
    smoke_model = smoke_optimizer = smoke_loss = smoke_train = smoke_val = None
    before = imgs = labels = vectors = val_loss = restored = handle = None
    gradient_checks = []
    result = dict(status='FAILED', configuration=cfg, baseline_configuration=cfg,
                  source_commit=SOURCE_COMMIT, dataset_identity=DATASET_IDENTITY,
                  environment=ENVIRONMENT, log=str(smoke_dir / 'smoke.log'))
    start = time.monotonic()
    with logged(smoke_dir / 'smoke.log'):
        try:
            seed_all(cfg['seed'])
            torch.cuda.reset_peak_memory_stats()
            print('Smoke configuration:', json.dumps(cfg, indent=2))
            smoke_model = make_model(cfg['gem_p'])
            smoke_optimizer = make_optimizer(smoke_model, cfg)
            smoke_loss = ContrastiveLoss(margin=cfg['loss_margin']).cuda()
            smoke_train, smoke_val = make_loaders(cfg)
            configure_original_loop(cfg)
            before = smoke_model.features[0].weight.detach().clone()
            def check_gradients(optimizer, args, kwargs):
                grads = [p.grad for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
                assert grads and all(torch.isfinite(g).all().item() for g in grads), 'Nonfinite gradients'
                gradient_checks.append(True)
            handle = smoke_optimizer.register_step_pre_hook(check_gradients)
            smoke_train_loss = original_train.train(smoke_train, smoke_model, smoke_loss, smoke_optimizer, 0)
            assert gradient_checks and math.isfinite(smoke_train_loss)
            assert not torch.equal(before, smoke_model.features[0].weight.detach()), 'No feature update'
            # One diagnostic validation tuple, original seven-image targets and loss.
            smoke_val.dataset.qsize = 1
            with torch.no_grad():
                smoke_val.dataset.create_epoch_tuples(smoke_model)
                imgs, labels = smoke_val.dataset[0]
                vectors = torch.cat([smoke_model(x.cuda()) for x in imgs], dim=1)
                val_loss = smoke_loss(vectors, labels.cuda())
            assert vectors.shape == (512, 7) and torch.isfinite(vectors).all()
            assert torch.allclose(vectors.norm(dim=0), torch.ones(7, device='cuda'), atol=1e-5)
            assert torch.isfinite(val_loss).item()
            # Separate smoke-only path, never latest.pt/best.pt or any full-run directory.
            smoke_checkpoint = smoke_dir / 'roundtrip.pt'
            temporary = smoke_dir / 'roundtrip.pt.tmp'
            torch.save({'state_dict': smoke_model.state_dict(), 'optimizer': smoke_optimizer.state_dict()}, temporary)
            temporary.replace(smoke_checkpoint)
            restored = torch.load(smoke_checkpoint, map_location='cpu', weights_only=True)
            assert all(torch.equal(v.cpu(), restored['state_dict'][k]) for k, v in smoke_model.state_dict().items())
            smoke_model.load_state_dict(restored['state_dict'], strict=True)
            smoke_optimizer.load_state_dict(restored['optimizer'])
            assert [g['lr'] for g in smoke_optimizer.param_groups] == [g['lr'] for g in restored['optimizer']['param_groups']]
            result.update(status='PASS', train_loss=smoke_train_loss, val_loss=val_loss.item(),
                          gradient_checks=len(gradient_checks), checkpoint=str(smoke_checkpoint),
                          train_mining=smoke_train.dataset.last_mining_diagnostic,
                          validation_mining=smoke_val.dataset.last_mining_diagnostic)
            print('SMOKE PASS: original mining, finite loss/gradients, parameter update and checkpoint round-trip checked.')
        except Exception as error:
            result.update(error_type=type(error).__name__, error=str(error))
            print('SMOKE FAILED:', result['error_type'], result['error'])
            traceback.print_exception(type(error), error, error.__traceback__)
            # Release upstream frames holding activation tensors; do not re-raise their traceback.
            traceback.clear_frames(error.__traceback__)
            error.__traceback__ = None
            print('No full training was started. Review this log; adjust the explicit pool cap for diversity failures. '
                  'For OOM, reduce resource settings explicitly or restart the kernel before retrying.')
        finally:
            if handle is not None:
                handle.remove()
            result['seconds'] = time.monotonic() - start
            result['peak_gpu_bytes'] = torch.cuda.max_memory_allocated()
            result['cuda_allocated_before_cleanup'] = torch.cuda.memory_allocated()
            smoke_model = smoke_optimizer = smoke_loss = smoke_train = smoke_val = None
            before = imgs = labels = vectors = val_loss = restored = handle = None
            gc.collect()
            torch.cuda.empty_cache()
            result['cuda_allocated_after_cleanup'] = torch.cuda.memory_allocated()
            print('GPU allocated bytes after smoke cleanup:', result['cuda_allocated_after_cleanup'])
            (smoke_dir / 'result.json').write_text(json.dumps(result, indent=2, allow_nan=False))
    return result

if RUN_SMOKE:
    smoke_outcome = run_smoke_test(resolved_config('smoke'))
    if smoke_outcome['status'] != 'PASS':
        # This new exception holds only a plain result dictionary, not model/activation frames.
        raise RuntimeError('Smoke failed; GPU cleanup completed. Details: ' + smoke_outcome['log']) from None
    SMOKE_RESULT = smoke_outcome
    print(json.dumps(SMOKE_RESULT, indent=2))
else:
    print('Smoke disabled. Configuration is ready; enable RUN_SMOKE explicitly after preflight.')


## 11–12. Full training controller, validation, logging, checkpoints, resume
RUN_FULL_TRAINING remains False in the delivered notebook. Enabling it runs the **fixed subset-100**, not the full SfM metadata. Fresh training must follow a successful smoke in this kernel; epoch-boundary resume uses an explicitly supplied trusted notebook checkpoint. The planned epoch count is a total target including completed epochs.

The controller preserves scheduler.step() before training and source loss/gradient accumulation. Checkpoints add scheduler state, RNG state, configuration, dataset identity, source commit/hashes and environment. Resume loads scheduler state rather than reconstructing/decaying it again. Only completed epochs are resumable; an interruption mid-epoch replays that epoch from the previous checkpoint. CUDA/library changes can prevent bitwise reproducibility.

Storage is bounded to latest.pt and best.pt, replaced atomically within this run folder. No unrelated files are deleted. A checkpoint is saved immediately after validation, before expensive benchmark evaluation, so evaluation failure does not discard the epoch. Best is chosen only by validation loss; benchmarks are never used to select it. Scheduler/seed changes and dropped residual tuples are listed in the run record. Finite gradients are checked before each update; no clipping or optimizer changes are introduced.


In [ ]:
def atomic_json(path, value):
    path = Path(path); temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False))
    temporary.replace(path)

def atomic_checkpoint(path, state):
    path = Path(path); temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(state, temporary)
    temporary.replace(path)

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all())

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])

def config_identity(cfg):
    # Total epoch target may be extended on resume; all other choices must match.
    return {k: v for k, v in cfg.items() if k != 'epochs'}

def assert_checkpoint_identity(state, cfg=None):
    assert state.get('format') == 'b0_sfm100_notebook_v1', 'Use a checkpoint produced by this notebook'
    assert state['source_commit'] == SOURCE_COMMIT
    assert state['source_hashes'] == SOURCE_HASHES
    assert state['dataset_identity'] == DATASET_IDENTITY, 'Dataset/weights identity changed'
    if cfg is not None:
        assert config_identity(state['config']) == config_identity(cfg), 'Run configuration changed on resume'

def checkpoint_state(model, optimizer, scheduler, epoch, best_loss, cfg, history):
    return dict(format='b0_sfm100_notebook_v1', epoch=epoch, meta=model.meta,
                state_dict=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                min_loss=best_loss, rng=rng_state(), config=cfg, dataset_identity=DATASET_IDENTITY,
                source_commit=SOURCE_COMMIT, source_hashes=SOURCE_HASHES, environment=ENVIRONMENT, history=history)

# This is the original single-scale extraction/ranking/AP protocol, with returned metrics.
def evaluate_legacy(model, image_size, destination, label):
    result = dict(label=label, image_size=image_size, scale=[1], whitening=False, query_expansion=False,
                  protocol='legacy_ok_junk', source_commit=SOURCE_COMMIT, datasets={})
    for name in ['oxford5k', 'paris6k']:
        bench = configdataset(name, str(DATA_ROOT / 'test'))
        images = [bench['im_fname'](bench, i) for i in range(bench['n'])]
        queries = [bench['qim_fname'](bench, i) for i in range(bench['nq'])]
        boxes = [tuple(g['bbx']) for g in bench['gnd']]
        gallery = imageretrievalnet.extract_vectors(model, images, image_size, transform, ms=[1])
        query = imageretrievalnet.extract_vectors(model, queries, image_size, transform, bbxs=boxes, ms=[1])
        assert gallery.shape == (512, bench['n']) and query.shape == (512, bench['nq'])
        assert torch.isfinite(gallery).all() and torch.isfinite(query).all()
        assert torch.allclose(gallery.norm(dim=0), torch.ones(bench['n']), atol=1e-4)
        assert torch.allclose(query.norm(dim=0), torch.ones(bench['nq']), atol=1e-4)
        ranks = np.argsort(-(gallery.numpy().T @ query.numpy()), axis=0)
        map_score, aps, _, _ = compute_map(ranks, bench['gnd'])
        assert np.isfinite(map_score) and np.isfinite(aps).all()
        result['datasets'][name] = dict(map=float(map_score), map_percent=float(map_score*100), aps=aps.tolist())
        print(name, 'legacy mAP:', f'{map_score*100:.4f}%')
        del gallery, query, ranks
    atomic_json(destination, result)
    return result

TRAINING_RESULT = None
if RUN_FULL_TRAINING:
    cfg = resolved_config('full')
    assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID), 'RUN_ID must be a simple folder name'
    run_dir = WORKSPACE / 'runs' / RUN_ID
    if RESUME_CHECKPOINT is None:
        assert SMOKE_RESULT is not None, 'Run a successful separate smoke test before fresh full training'
        assert all(SMOKE_RESULT['baseline_configuration'][key] == cfg[key] for key in
                   ['image_size', 'batch_size', 'optimizer', 'lr', 'weight_decay', 'momentum', 'loss_margin', 'gem_p', 'seed', 'workers']), 'Core settings changed since smoke; repeat smoke'
        if run_dir.exists() and any(run_dir.iterdir()):
            raise RuntimeError('Run directory is not empty. Choose a new RUN_ID or resume explicitly.')
    else:
        if not Path(RESUME_CHECKPOINT).is_file():
            raise RuntimeError('Resume checkpoint is missing: ' + str(RESUME_CHECKPOINT))
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_all(cfg['seed'])
    model = make_model(cfg['gem_p'])
    optimizer = make_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=math.exp(-0.01))
    criterion = ContrastiveLoss(margin=cfg['loss_margin']).cuda()
    train_loader, val_loader = make_loaders(cfg)
    configure_original_loop(cfg)
    start_epoch, best_loss, history = 0, float('inf'), []
    if RESUME_CHECKPOINT is not None:
        # Explicit pickle load is needed for our RNG state; use only your own trusted run artifacts.
        resumed = torch.load(RESUME_CHECKPOINT, map_location='cpu', weights_only=False)
        assert_checkpoint_identity(resumed, cfg)
        model.load_state_dict(resumed['state_dict'], strict=True)
        optimizer.load_state_dict(resumed['optimizer'])
        scheduler.load_state_dict(resumed['scheduler'])
        start_epoch, best_loss, history = resumed['epoch'], resumed['min_loss'], resumed['history']
        restore_rng(resumed['rng'])
        # Preserve the best state when continuing into a new run folder from an uploaded checkpoint.
        if not (run_dir / 'best.pt').exists():
            if resumed['epoch'] == min(history, key=lambda row: row['val_loss'])['epoch']:
                atomic_checkpoint(run_dir / 'best.pt', resumed)
            else:
                sibling_best = Path(RESUME_CHECKPOINT).parent / 'best.pt'
                if not sibling_best.is_file():
                    raise RuntimeError('Attach the matching best.pt alongside latest.pt to preserve model selection.')
                best_state = torch.load(sibling_best, map_location='cpu', weights_only=False)
                assert_checkpoint_identity(best_state, cfg)
                assert best_state['min_loss'] == best_loss
                assert best_state['epoch'] == min(history, key=lambda row: row['val_loss'])['epoch']
                atomic_checkpoint(run_dir / 'best.pt', best_state)
        if (run_dir / 'best.pt').exists():
            best_state = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
            assert_checkpoint_identity(best_state, cfg)
            assert best_state['epoch'] == min(history, key=lambda row: row['val_loss'])['epoch']
            assert best_state['min_loss'] == best_loss
            del best_state
        del resumed
    assert 0 <= start_epoch <= cfg['epochs']
    record = dict(config=cfg, decision_note=FULL_DECISION_NOTE, source_commit=SOURCE_COMMIT, source_hashes=SOURCE_HASHES,
                  dataset_identity=DATASET_IDENTITY, paths=PATHS, environment=ENVIRONMENT,
                  deviations_from_defaults={k: [SOURCE_DEFAULTS[k], v] for k,v in cfg.items() if v != SOURCE_DEFAULTS[k]},
                  actual_train_pool=train_loader.dataset.poolsize, actual_train_queries=train_loader.dataset.qsize,
                  processed_train_tuples=len(train_loader)*cfg['batch_size'], processed_val_tuples=len(val_loader)*cfg['batch_size'],
                  subset='fixed k100 seed42; active metadata only', resume=str(RESUME_CHECKPOINT),
                  scheduler_gamma=math.exp(-0.01), scheduler_timing='before train',
                  checkpoint_policy='atomic latest+best; epoch-boundary resume; checkpoint before benchmark eval',
                  mining_workers=8, precision='FP32', smoke=SMOKE_RESULT)
    atomic_json(run_dir / 'run_record.json', record)
    def finite_gradients(optimizer, args, kwargs):
        grads = [p.grad for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
        if not grads or not all(torch.isfinite(g).all().item() for g in grads):
            raise FloatingPointError('Nonfinite/absent gradients; optimizer update aborted')
    grad_handle = optimizer.register_step_pre_hook(finite_gradients)
    try:
        with logged(run_dir / 'training.log'):
            print('Starting/resuming epoch:', start_epoch, 'LR:', [g['lr'] for g in optimizer.param_groups])
            if start_epoch == 0:
                evaluate_legacy(model, cfg['eval_image_size'], run_dir / 'evaluation_initial.json', 'initial_caffe')
            for epoch in range(start_epoch, cfg['epochs']):
                seed_all(cfg['seed'] + epoch)
                start = time.monotonic()
                torch.cuda.reset_peak_memory_stats()
                scheduler.step()  # Preserve the historical timing; modern torch may warn on first step.
                learning_rates = [g['lr'] for g in optimizer.param_groups]
                print('Epoch', epoch+1, 'seed', cfg['seed']+epoch, 'LR before update', learning_rates)
                train_loss = original_train.train(train_loader, model, criterion, optimizer, epoch)
                with torch.no_grad():
                    val_loss = original_train.validate(val_loader, model, criterion, epoch)
                assert math.isfinite(train_loss) and math.isfinite(val_loss)
                improved = val_loss < best_loss
                best_loss = min(best_loss, val_loss)
                history.append(dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
                                    lr=learning_rates, gem_p=model.pool.p.item(), seconds_train_val=time.monotonic()-start,
                                    peak_gpu_bytes=torch.cuda.max_memory_allocated()))
                state = checkpoint_state(model, optimizer, scheduler, epoch+1, best_loss, cfg, history)
                atomic_checkpoint(run_dir / 'latest.pt', state)
                if improved:
                    atomic_checkpoint(run_dir / 'best.pt', state)
                atomic_json(run_dir / 'history.json', history)
                print('Checkpoint saved, best validation loss:', best_loss)
                if (epoch+1) % cfg['test_freq'] == 0:
                    evaluate_legacy(model, cfg['eval_image_size'], run_dir / f'evaluation_epoch_{epoch+1:03d}.json', f'epoch_{epoch+1}')
                print('Total epoch seconds including evaluation:', time.monotonic()-start)
                del state
            TRAINING_RESULT = dict(completed_epochs=len(history), best_validation_loss=best_loss,
                                   latest=str(run_dir / 'latest.pt'), best=str(run_dir / 'best.pt'))
            atomic_json(run_dir / 'training_result.json', TRAINING_RESULT)
    finally:
        grad_handle.remove()
        del model, optimizer, scheduler, criterion, train_loader, val_loader
        gc.collect(); torch.cuda.empty_cache()
else:
    print('Full training disabled. No optimizer updates or benchmark evaluation are launched by this cell.')


## 13. Final legacy mAP evaluation — explicit opt-in
Enable RUN_FINAL_EVALUATION to evaluate a trusted notebook-generated checkpoint. By default it loads the run's **best validation-loss** checkpoint, not the last model or smoke model. A selected path is permitted under /kaggle/input for a resumed/offline session. Both benchmarks use original query bounding boxes and gallery order, dot-product ranking and upstream trapezoidal AP with junk removal. The checkpoint SHA-256 is saved alongside results. No whitening or QE code is called.


In [ ]:
FINAL_RESULTS = None
if RUN_FINAL_EVALUATION:
    assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
    checkpoint = Path(EVALUATION_CHECKPOINT) if EVALUATION_CHECKPOINT else WORKSPACE / 'runs' / RUN_ID / 'best.pt'
    if not checkpoint.is_file():
        raise RuntimeError('No trained checkpoint available for final evaluation: ' + str(checkpoint))
    state = torch.load(checkpoint, map_location='cpu', weights_only=False)  # Trusted notebook artifact only.
    assert_checkpoint_identity(state)
    evaluation_cfg = state['config']
    eval_model = make_model(evaluation_cfg['gem_p'])
    eval_model.load_state_dict(state['state_dict'], strict=True)
    out = WORKSPACE / 'evaluations' / (RUN_ID + '_' + str(time.time_ns()))
    out.mkdir(parents=True)
    with logged(out / 'evaluation.log'):
        FINAL_RESULTS = evaluate_legacy(eval_model, evaluation_cfg['eval_image_size'], out / 'metrics.json',
                                        'checkpoint_epoch_' + str(state['epoch']))
        FINAL_RESULTS.update(checkpoint=str(checkpoint), checkpoint_sha256=sha256_file(checkpoint),
                             checkpoint_epoch=state['epoch'], dataset_identity=DATASET_IDENTITY, environment=ENVIRONMENT)
        atomic_json(out / 'metrics.json', FINAL_RESULTS)
    del state, eval_model
    gc.collect(); torch.cuda.empty_cache()
else:
    print('Final evaluation disabled; no mAP result is claimed.')


## 14. Results, artifacts, and deviation ledger

**Method preserved:** original VGG16 feature stack (last classification max-pool omitted), shared scalar GeM, L2-normalized 512-D descriptor, seven-image contrastive tuples, random pair/pool selection and hard-negative ranking, distinct negative clusters, source train/validation loops and summed loss/gradient accumulation. Preprocessing remains source RGB/mean/std, without augmentation. Single-scale legacy evaluation preserves bbox cropping and AP/junk logic.

**Fixed methodological deviations:** the user-specified subset-100 replaces full training data; VGG16 replaces CLI ResNet101; legacy Oxford/Paris replaces training CLI revisited datasets. All local/network/descriptor whitening, regional pooling and QE are off. Any image/query/pool/epoch/LR/optimizer/seed/GeM choices differing from executable defaults are recorded automatically and require an explicit decision note. No automatic OOM fallback is performed.

**Compatibility/operational changes:** two existing B0 Pillow/NumPy API substitutions; offline pinned source setup; in-memory data-root routing and symlinks; strict local weight transport instead of load_url; modern explicit torch.load behavior; finite-value checks; scheduler/RNG/config identity added to checkpoints; atomic latest/best retention instead of one archive per epoch; epoch checkpoint saved before benchmark evaluation. Scheduler timing remains pre-epoch; its actual LR is logged. The original drop_last behavior is retained, including validation residuals. Mining/evaluation still use the upstream eight-worker loaders.

**Smoke-only deviations:** one training batch, bounded mining pool, one validation tuple and a 128-pixel initialization probe. Smoke weights are discarded; their timings are not full-epoch estimates.

Smoke diagnostics preview the exact pool with an independent CPU generator without advancing the original RNG. Smoke failures are logged and GPU resources released in finally; source mining code is unchanged.

Download the run folder and matching **latest.pt + best.pt** before ending a session. To resume, attach those files as another input, set an exact RESUME_CHECKPOINT path, retain matching choices/dataset/source identity, and optionally extend epochs. Use a new RUN_ID when importing into a fresh working session. This supports epoch-boundary continuation only; interrupted work after the most recent completed epoch is replayed. It does not claim exact cross-version or cross-device reproducibility.

The preflight counts/provenance/hash checks identify the attached subset; no independent source hash for the historical seed-42 draw was supplied. Neither full data preparation nor successful Kaggle training/evaluation is claimed by the delivered notebook.


In [ ]:
print('Workspace:', WORKSPACE)
print('Source commit:', SOURCE_COMMIT)
print('Dataset metadata SHA-256:', DATASET_IDENTITY['metadata_sha256'])
print('Smoke result:', SMOKE_RESULT)
print('Training result:', TRAINING_RESULT)
print('Final result:', FINAL_RESULTS)
for folder in ['runs', 'evaluations']:
    root = WORKSPACE / folder
    if root.exists():
        for path in sorted(root.rglob('*')):
            if path.is_file():
                print(path.relative_to(WORKSPACE), path.stat().st_size, 'bytes')
